# Notebook 02 — Plane-Specific Representation Learning with RICA

**Stage:** Frozen MedDINOv3 features → three plane-specific RICA modules.

## Goal

Load the features already extracted in Notebook 01 and implement the RICA stage for axial, coronal, and sagittal views.

Each view enters as `(B, D, E)`, where the current extraction has `D=64` slices and `E=768` features per slice. The paper describes treating this sequence as a pseudo-2D feature map and applying RICA independently to each plane.

**Reconstruction note:** the main paper does not fully specify the pseudo-2D channel layout or every RICA hyperparameter. This notebook uses `(B,D,E) → (B,1,D,E)`, one RICA block per plane, two 3×3 convolution layers in the residual branch, and Coordinate Attention plus a 1×1 projection in the shortcut. These are transparent implementation choices consistent with the described method and cited RICAU-Net block, not verified original author source code.

**Training note:** this notebook builds and validates trainable RICA modules. It does not claim RICA has learned useful representations. In the paper, RICA is part of the downstream classification architecture and should receive its learning signal during full-model training.

## 1. Environment and reproducibility

In [1]:
import os, json, random, platform
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.18
PyTorch: 2.11.0+cpu
Device: cpu


## 2. Locate and load the Notebook 01 features

Edit `FEATURE_DIR` if the automatic Kaggle paths do not match your dataset.

In [2]:
from pathlib import Path

BASE_DIR = Path(
    "/kaggle/input/notebooks/kamifiza/axial-centric-cross-plane-attention-for-3d-medical"
)

print("Files and folders:")
for item in BASE_DIR.iterdir():
    print(item.name, " [DIR]" if item.is_dir() else " [FILE]")
    FEATURE_DIR = BASE_DIR / "features"

print("Feature files:")
for item in FEATURE_DIR.iterdir():
    print(item.name)

Files and folders:
MedDINOv3  [DIR]
__notebook__.ipynb  [FILE]
__output__.json  [FILE]
__results__.html  [FILE]
__results___files  [DIR]
custom.css  [FILE]
features  [DIR]
meddinov3_ckpt  [DIR]
medmnist  [DIR]
Feature files:
features_test.pt
features_train.pt
features_val.pt
meta.json


In [3]:
import torch

FEATURE_DIR = BASE_DIR / "features"

train_data = torch.load(
    FEATURE_DIR / "features_train.pt",
    map_location="cpu",
    weights_only=False
)

print("Available keys:", train_data.keys())

Available keys: dict_keys(['axial_features', 'coronal_features', 'sagittal_features', 'labels', 'volume_index', 'slice_indices'])


### 2.1 Load feature tensors

Only load feature files that you trust. `weights_only=False` is used because the Notebook 01 files contain dictionaries of tensors and metadata.

In [4]:
def load_split(path):
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(path, map_location="cpu")
    if not isinstance(obj, dict):
        raise TypeError(f"Expected dictionary in {path}, got {type(obj)}")
    return obj

splits = {
    "train": load_split(FEATURE_DIR / "features_train.pt"),
    "val": load_split(FEATURE_DIR / "features_val.pt"),
    "test": load_split(FEATURE_DIR / "features_test.pt"),
}
for name, data in splits.items():
    print(name, list(data.keys()))

train ['axial_features', 'coronal_features', 'sagittal_features', 'labels', 'volume_index', 'slice_indices']
val ['axial_features', 'coronal_features', 'sagittal_features', 'labels', 'volume_index', 'slice_indices']
test ['axial_features', 'coronal_features', 'sagittal_features', 'labels', 'volume_index', 'slice_indices']


## 3. Validate the extracted features

In [5]:
PLANE_KEYS = {
    "axial": "axial_features",
    "coronal": "coronal_features",
    "sagittal": "sagittal_features",
}
EXPECTED_COUNTS = {"train": 971, "val": 161, "test": 610}
D, E = 64, 768

for split_name, data in splits.items():
    labels = torch.as_tensor(data["labels"]).view(-1)
    assert len(labels) == EXPECTED_COUNTS[split_name], (split_name, len(labels))
    for plane, key in PLANE_KEYS.items():
        x = torch.as_tensor(data[key])
        assert x.shape == (EXPECTED_COUNTS[split_name], D, E), (split_name, plane, x.shape)
        assert torch.isfinite(x).all(), f"Non-finite values: {split_name}/{plane}"
        print(f"{split_name:5s} | {plane:8s} | {tuple(x.shape)} | {x.dtype}")
print("All feature checks passed.")

train | axial    | (971, 64, 768) | torch.float32
train | coronal  | (971, 64, 768) | torch.float32
train | sagittal | (971, 64, 768) | torch.float32
val   | axial    | (161, 64, 768) | torch.float32
val   | coronal  | (161, 64, 768) | torch.float32
val   | sagittal | (161, 64, 768) | torch.float32
test  | axial    | (610, 64, 768) | torch.float32
test  | coronal  | (610, 64, 768) | torch.float32
test  | sagittal | (610, 64, 768) | torch.float32
All feature checks passed.


## 4. RICA block implementation

The cited RICAU-Net description motivates a residual branch with two convolution layers and a shortcut containing Coordinate Attention (CA) followed by a projection shortcut (1×1 convolution + BatchNorm).

Coordinate Attention pools along the two spatial axes separately, then produces direction-aware gates. The implementation below follows the common Coordinate Attention design.

In [6]:
class HSwish(nn.Module):
    def forward(self, x):
        return x * F.relu6(x + 3.0, inplace=True) / 6.0


class CoordinateAttention(nn.Module):
    # Directional pooling CA for 2D feature maps: input/output (B,C,H,W).
    def __init__(self, channels, reduction=32):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.conv1 = nn.Conv2d(channels, hidden, 1, bias=True)
        self.bn1 = nn.BatchNorm2d(hidden)
        self.act = HSwish()
        self.conv_h = nn.Conv2d(hidden, channels, 1, bias=True)
        self.conv_w = nn.Conv2d(hidden, channels, 1, bias=True)

    def forward(self, x):
        identity = x
        _, _, h, w = x.shape
        x_h = x.mean(dim=3, keepdim=True)                       # B,C,H,1
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)   # B,C,W,1
        y = torch.cat([x_h, x_w], dim=2)                        # B,C,H+W,1
        y = self.act(self.bn1(self.conv1(y)))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)
        a_h = torch.sigmoid(self.conv_h(y_h))
        a_w = torch.sigmoid(self.conv_w(y_w))
        return identity * a_h * a_w


class RICAResidualBlock(nn.Module):
    # Residual path: Conv-BN-ReLU twice. Shortcut: CA -> 1x1 projection -> BN.
    def __init__(self, in_channels=1, mid_channels=1, out_channels=1,
                 kernel_size=3, ca_reduction=32):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size,
                               padding=padding, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(mid_channels, out_channels, kernel_size,
                               padding=padding, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)

        self.ca = CoordinateAttention(in_channels, reduction=ca_reduction)
        self.projection = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        residual = self.relu1(self.bn1(self.conv1(x)))
        residual = self.relu2(self.bn2(self.conv2(residual)))
        shortcut = self.projection(self.ca(x))
        if residual.shape != shortcut.shape:
            raise RuntimeError(f"RICA branch mismatch: {residual.shape} vs {shortcut.shape}")
        return residual + shortcut

## 5. Create independent RICA modules for the three planes

The current reconstruction adds a singleton channel: `(B,D,E) → (B,1,D,E)`. Each plane has its own RICA parameters; the three modules do not share weights.

In [7]:
class PlaneRICA(nn.Module):
    def __init__(self, mid_channels=1, kernel_size=3, ca_reduction=32):
        super().__init__()
        self.block = RICAResidualBlock(
            in_channels=1, mid_channels=mid_channels, out_channels=1,
            kernel_size=kernel_size, ca_reduction=ca_reduction
        )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(f"Expected (B,D,E), received {tuple(x.shape)}")
        return self.block(x.unsqueeze(1)).squeeze(1)


class ThreePlaneRICA(nn.Module):
    def __init__(self, mid_channels=1, kernel_size=3, ca_reduction=32):
        super().__init__()
        self.rica_axial = PlaneRICA(mid_channels, kernel_size, ca_reduction)
        self.rica_coronal = PlaneRICA(mid_channels, kernel_size, ca_reduction)
        self.rica_sagittal = PlaneRICA(mid_channels, kernel_size, ca_reduction)

    def forward(self, axial, coronal, sagittal):
        return {
            "axial": self.rica_axial(axial),
            "coronal": self.rica_coronal(coronal),
            "sagittal": self.rica_sagittal(sagittal),
        }


RICA_CONFIG = {
    "pseudo_2d_layout": "(B,D,E) -> (B,1,D,E)",
    "in_channels":  1,
    "mid_channels": 1,
    "out_channels": 1,
    "kernel_size": 3,
    "ca_reduction": 32,
    "blocks_per_plane": 1,
    "plane_weights_shared": False,
    "status": "reconstruction; not verified original author code",
    "training_status": "initialized, not trained in this notebook"
}

rica_model = ThreePlaneRICA(
    mid_channels=RICA_CONFIG["mid_channels"],
    kernel_size=RICA_CONFIG["kernel_size"],
    ca_reduction=RICA_CONFIG["ca_reduction"]
).to(DEVICE)

print(rica_model)

ThreePlaneRICA(
  (rica_axial): PlaneRICA(
    (block): RICAResidualBlock(
      (conv1): Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): ReLU(inplace=True)
      (conv2): Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): ReLU(inplace=True)
      (ca): CoordinateAttention(
        (conv1): Conv2d(1, 8, kernel_size=(1, 1), stride=(1, 1))
        (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): HSwish()
        (conv_h): Conv2d(8, 1, kernel_size=(1, 1), stride=(1, 1))
        (conv_w): Conv2d(8, 1, kernel_size=(1, 1), stride=(1, 1))
      )
      (projection): Sequential(
        (0): Conv2d(1, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(1, ep

## 6. Parameter count and independent-weight check

The parameter count describes this chosen configuration only. It cannot establish that the paper used these exact hidden channel dimensions.

In [8]:
def count_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

for name in ["rica_axial", "rica_coronal", "rica_sagittal"]:
    print(name, count_parameters(getattr(rica_model, name)), "trainable parameters")
print("Total:", count_parameters(rica_model))

param_ids = {
    name: {id(p) for p in getattr(rica_model, name).parameters()}
    for name in ["rica_axial", "rica_coronal", "rica_sagittal"]
}
assert param_ids["rica_axial"].isdisjoint(param_ids["rica_coronal"])
assert param_ids["rica_axial"].isdisjoint(param_ids["rica_sagittal"])
assert param_ids["rica_coronal"].isdisjoint(param_ids["rica_sagittal"])
print("Independent plane parameters: PASS")

rica_axial 75 trainable parameters
rica_coronal 75 trainable parameters
rica_sagittal 75 trainable parameters
Total: 225
Independent plane parameters: PASS


## 7. Forward-pass smoke test

Run a small batch from the real cached features.

In [9]:
BATCH_SIZE = 4
train_data = splits["train"]
batch = {
    plane: torch.as_tensor(train_data[key][:BATCH_SIZE], dtype=torch.float32).to(DEVICE)
    for plane, key in PLANE_KEYS.items()
}

rica_model.eval()
with torch.no_grad():
    outputs = rica_model(batch["axial"], batch["coronal"], batch["sagittal"])

for plane in PLANE_KEYS:
    print(plane, "input:", tuple(batch[plane].shape),
          "output:", tuple(outputs[plane].shape),
          "finite:", bool(torch.isfinite(outputs[plane]).all()))
    assert outputs[plane].shape == batch[plane].shape
    assert torch.isfinite(outputs[plane]).all()
print("Forward test: PASS")

axial input: (4, 64, 768) output: (4, 64, 768) finite: True
coronal input: (4, 64, 768) output: (4, 64, 768) finite: True
sagittal input: (4, 64, 768) output: (4, 64, 768) finite: True
Forward test: PASS


## 8. Gradient-flow test

This uses a synthetic scalar objective only to verify autograd. It is **not** training and says nothing about classification quality.

In [10]:

import copy

# Create a copy so the original RICA model remains unchanged
gradient_test_model = copy.deepcopy(rica_model).to(DEVICE)

# Use the actual loaded training features
gradient_test_inputs = {
    "axial": train_data["axial_features"][:2].float().to(DEVICE),
    "coronal": train_data["coronal_features"][:2].float().to(DEVICE),
    "sagittal": train_data["sagittal_features"][:2].float().to(DEVICE),
}

# Test gradient flow on the copied model
gradient_test_model.train()
gradient_test_model.zero_grad(set_to_none=True)

out = gradient_test_model(
    gradient_test_inputs["axial"],
    gradient_test_inputs["coronal"],
    gradient_test_inputs["sagittal"]
)

test_loss = sum(v.square().mean() for v in out.values())
test_loss.backward()

# Verify all three independent RICA modules
for name in ["rica_axial", "rica_coronal", "rica_sagittal"]:
    module = getattr(gradient_test_model, name)
    grads = [
        p.grad for p in module.parameters()
        if p.requires_grad
    ]

    assert any(g is not None for g in grads)
    assert all(
        torch.isfinite(g).all()
        for g in grads if g is not None
    )

print("Gradient flow test passed for all three RICA modules.")


Gradient flow test passed for all three RICA modules.


## 9. Train / validation / test shape smoke tests

No model fitting or hyperparameter selection is performed on validation or test data.

In [11]:
rica_model.eval()
with torch.no_grad():
    for split_name, data in splits.items():
        sample = {
            plane: torch.as_tensor(data[key][:2], dtype=torch.float32).to(DEVICE)
            for plane, key in PLANE_KEYS.items()
        }
        pred = rica_model(sample["axial"], sample["coronal"], sample["sagittal"])
        for plane in PLANE_KEYS:
            assert pred[plane].shape == sample[plane].shape
            assert torch.isfinite(pred[plane]).all()
        print(split_name, "PASS")

train PASS
val PASS
test PASS


## 10. Save the RICA initialization and configuration

This checkpoint is an **untrained initialization**, not learned features. The full architecture should load these weights and train RICA jointly with the subsequent trainable stages.

In [12]:

import json
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/rica_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = OUTPUT_DIR / "rica_initialization.pt"
config_path = OUTPUT_DIR / "rica_config.json"

# Load the actual feature metadata directly from disk
with open(FEATURE_DIR / "meta.json", "r") as f:
    feature_metadata = json.load(f)

# Save the original RICA model, not the gradient-test copy
torch.save({
    "state_dict": rica_model.state_dict(),
    "config": RICA_CONFIG,
    "seed": SEED,
    "feature_metadata": feature_metadata,
    "training_status": "untrained initialization"
}, checkpoint_path)

with open(config_path, "w") as f:
    json.dump(RICA_CONFIG, f, indent=2)

print("Saved:", checkpoint_path)
print("Saved:", config_path)
print("Training status: untrained initialization")


Saved: /kaggle/working/rica_artifacts/rica_initialization.pt
Saved: /kaggle/working/rica_artifacts/rica_config.json
Training status: untrained initialization
